In [ ]:
# ==========================================================
#  STEP 1 — Install dependencies
# ==========================================================
!pip install roboflow ultralytics opencv-python matplotlib --quiet

In [ ]:
# ==========================================================
# STEP 2 — Download your Roboflow dataset
# ==========================================================
import os
from roboflow import Roboflow

# Initialize Roboflow using your API key
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])

# Access your specific project and dataset version
project = rf.workspace("ahmed-hany-rf9xi").project("draga-sb2rn")
dataset = project.version(1).download("yolov12")


In [ ]:
# ==========================================================
# 🧠 STEP 3 — Train YOLOv8 model on the dataset
# ==========================================================
from ultralytics import YOLO

# Load a small pretrained YOLOv8 model (fast and lightweight)
model = YOLO("yolov8n.pt")

# Train the model using your dataset
model.train(
    data=f"{dataset.location}/data.yaml",  # YOLO dataset configuration
    epochs=50,                            # training epochs
    imgsz=640,                            # image size for training
    batch=8                               # batch size
)

# Save trained weights
model.save("drug_detector.pt")

In [ ]:
# ==========================================================
# 🧠 STEP 4 — Test model on a random test image
# ==========================================================
import cv2
import matplotlib.pyplot as plt
import os, random

# Pick random test image
image_folder = os.path.join(dataset.location, "test", "images")
image_file = random.choice(os.listdir(image_folder))
image_path = os.path.join(image_folder, image_file)

# Load image
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Run YOLO detection
results = model(image_rgb)

# Draw detection boxes on the image
annotated_image = results[0].plot()  # YOLOv8 provides a built-in plotting method

# Show result
plt.figure(figsize=(10, 10))
plt.imshow(annotated_image)
plt.axis("off")
plt.title(f"Detected objects in: {image_file}")
plt.show()


# **We run this only for using the model without training again**

In [ ]:
!pip install roboflow ultralytics opencv-python matplotlib --quiet
# ==========================================================
# 🧠 STEP 4 — Load trained model and test it on your dataset >>> run model without train
# ==========================================================
import cv2
import matplotlib.pyplot as plt
import os, random
from ultralytics import YOLO
from roboflow import Roboflow

# 🟩 1️⃣ Download your dataset from Roboflow (same one you trained on)
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("ahmed-hany-rf9xi").project("draga-sb2rn")
dataset = project.version(1).download("yolov8")

# 🟩 2️⃣ Load your trained YOLO model (the .pt file you trained or downloaded)
model = YOLO("/content/drug_detector.pt")  # replace with your model filename if different

# 🟩 3️⃣ Pick a random test image from the same dataset
image_folder = os.path.join(dataset.location, "test", "images")
image_file = random.choice(os.listdir(image_folder))
image_path = os.path.join(image_folder, image_file)

# 🟩 4️⃣ Load and prepare the image
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# 🟩 5️⃣ Run object detection using the trained model
results = model(image_rgb)

# 🟩 6️⃣ Draw and visualize detection boxes
annotated_image = results[0].plot()  # YOLOv8 built-in method for visualization

# 🟩 7️⃣ Show the detected result
plt.figure(figsize=(10, 10))
plt.imshow(annotated_image)
plt.axis("off")
plt.title(f"Detected objects in: {image_file}")
plt.show()


In [ ]:
# Cell A — install OCR libs (append this cell)
!pip install easyocr pytesseract imutils --quiet
# Note: pytesseract requires Tesseract installed on your system.
# On Linux you can: sudo apt-get install -y tesseract-ocr
# On Windows, download from https://github.com/tesseract-ocr/tesseract and ensure PATH includes tesseract.exe


In [ ]:
# # Cell B — OCR pipeline: crop, enhance, deskew, run EasyOCR + pytesseract, append results
# import cv2
# import numpy as np
# import pandas as pd
# import easyocr
# import pytesseract
# import math
# from imutils import rotate_bound

# # Initialize EasyOCR once (use GPU if available by adding gpu=True)
# _reader = easyocr.Reader(['en'], gpu=False)

# def enhance_image_for_ocr(img):
#     """Enhance crop: convert, denoise, CLAHE, upscale"""
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img.copy()
#     # Denoise
#     gray = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)
#     # CLAHE
#     clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
#     gray = clahe.apply(gray)
#     # Resize (upscale) to help OCR on small text
#     h, w = gray.shape
#     scale = max(1, int(640 / max(h, w)))  # upscale so largest side ~640
#     if scale > 1:
#         gray = cv2.resize(gray, (w*scale, h*scale), interpolation=cv2.INTER_LINEAR)
#     return gray

# def try_rotations_and_ocr(crop):
#     """
#     Try multiple small rotations to handle rotated labels. For each rotation
#     run EasyOCR and pytesseract and pick the best text by confidence.
#     Returns (best_text, best_confidence, source_engine).
#     """
#     best_text = ""
#     best_conf = -1.0
#     best_engine = None

#     angles = [0, -90, 90] + list(range(-30, 31, 5))  # include major orientations + fine sweep
#     for angle in angles:
#         rotated = rotate_bound(crop, angle) if angle != 0 else crop
#         proc = enhance_image_for_ocr(rotated)

#         # EasyOCR
#         try:
#             e_res = _reader.readtext(proc, detail=1)  # returns list of (bbox, text, conf)
#             if e_res:
#                 # pick highest-confidence detection (there may be multiple small text pieces)
#                 best_e = max(e_res, key=lambda r: r[2])
#                 e_text = best_e[1].strip()
#                 e_conf = float(best_e[2])
#                 if e_text and e_conf > best_conf:
#                     best_text = e_text
#                     best_conf = e_conf
#                     best_engine = "easyocr"
#         except Exception:
#             pass

#         # pytesseract (try different psm modes)
#         try:
#             # psm 7 = treat as a single text line; psm 6 = block of text
#             for psm in ("7", "6"):
#                 config = f'--psm {psm} --oem 3 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789-_.'
#                 t_text = pytesseract.image_to_string(proc, config=config).strip()
#                 # get conf values (pytesseract image_to_data)
#                 dt = pytesseract.image_to_data(proc, config=config, output_type=pytesseract.Output.DICT)
#                 # average confidence of words (dt['conf'] sometimes contains '-1' for blanks)
#                 confs = [int(x) for x in dt['conf'] if x.strip().lstrip('-').isdigit()]
#                 t_conf = (sum(confs)/len(confs)) if confs else 0
#                 if t_text and t_conf > best_conf:
#                     best_text = t_text
#                     best_conf = float(t_conf)
#                     best_engine = f"pytesseract_psm{psm}"
#         except Exception:
#             pass

#     # normalize confidence to 0..1 scale when coming from pytesseract (which uses 0..100)
#     if best_engine and best_engine.startswith("pytesseract"):
#         best_conf = best_conf / 100.0

#     return best_text, best_conf, best_engine

# def ocr_results_from_ultralytics(results, original_image, save_csv_path=None):
#     """
#     results: ultralytics results for one image (results[0])
#     original_image: the rgb image array (image_rgb)
#     Returns a DataFrame with bbox, class, score, ocr_text, ocr_conf, ocr_engine
#     """
#     rows = []
#     # ultralytics stores boxes in results.boxes (xyxy), and .cls and .conf
#     boxes = results.boxes.cpu().numpy() if hasattr(results.boxes, "cpu") else None

#     # Fallback: try reading from results.boxes.data if above not available
#     if boxes is None:
#         try:
#             # For newer ultralytics versions: results.boxes.xyxy, results.boxes.conf, results.boxes.cls
#             xyxy = results.boxes.xyxy.numpy()
#             confs = results.boxes.conf.numpy()
#             clses = results.boxes.cls.numpy()
#         except Exception:
#             # Try attribute-based extraction
#             xyxy, confs, clses = [], [], []
#             for b in results.boxes:
#                 xy = b.xyxy.cpu().numpy().tolist()[0]
#                 xyxy.append(xy)
#                 confs.append(float(b.conf.cpu().numpy()))
#                 clses.append(int(b.cls.cpu().numpy()))
#             xyxy = np.array(xyxy)
#             confs = np.array(confs)
#             clses = np.array(clses)
#     else:
#         # If we have raw numpy boxes, assume format [x1,y1,x2,y2,conf,class] or similar
#         # Better to use results.boxes.xyxy, results.boxes.conf, results.boxes.cls when available
#         try:
#             xyxy = results.boxes.xyxy.cpu().numpy()
#             confs = results.boxes.conf.cpu().numpy()
#             clses = results.boxes.cls.cpu().numpy()
#         except Exception:
#             # last resort: interpret boxes
#             xyxy = boxes[:, :4]
#             confs = boxes[:, 4] if boxes.shape[1] >= 5 else np.zeros(len(boxes))
#             clses = boxes[:, 5] if boxes.shape[1] >= 6 else np.zeros(len(boxes))

#     h_img, w_img = original_image.shape[:2]

#     for i, (box, conf, cls) in enumerate(zip(xyxy, confs, clses)):
#         x1, y1, x2, y2 = [int(max(0, v)) for v in box[:4]]
#         # padding to include a bit around label
#         pad_x = int(0.03 * (x2 - x1) + 2)
#         pad_y = int(0.03 * (y2 - y1) + 2)
#         xa = max(0, x1 - pad_x); ya = max(0, y1 - pad_y)
#         xb = min(w_img, x2 + pad_x); yb = min(h_img, y2 + pad_y)

#         crop = original_image[ya:yb, xa:xb].copy()
#         if crop.size == 0:
#             ocr_text, ocr_conf, ocr_engine = "", 0.0, None
#         else:
#             ocr_text, ocr_conf, ocr_engine = try_rotations_and_ocr(crop)

#         rows.append({
#             "idx": i,
#             "bbox": (xa, ya, xb, yb),
#             "class": int(cls),
#             "box_conf": float(conf),
#             "ocr_text": ocr_text,
#             "ocr_confidence": float(ocr_conf),
#             "ocr_engine": ocr_engine
#         })

#     df = pd.DataFrame(rows)
#     if save_csv_path:
#         df.to_csv(save_csv_path, index=False)
#     return df

# # Example usage (append this after you run detection on an image):
# # df_ocr = ocr_results_from_ultralytics(results[0], image_rgb, save_csv_path="ocr_results.csv")
# # display(df_ocr)
# # You can append results to your dataset or the detections list by saving df_ocr or merging.


In [ ]:
# # ==========================================================
# # TEST OCR on a single image and show results (text on top)
# # ==========================================================
# import matplotlib.pyplot as plt
# import random

# # Pick a random test image from your dataset
# image_folder = os.path.join(dataset.location, "test", "images")
# image_file = random.choice(os.listdir(image_folder))
# image_path = os.path.join(image_folder, image_file)

# # Load image
# image = cv2.imread(image_path)
# image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# # Run YOLO detection
# results = model(image_rgb)

# # Run OCR on detected boxes
# df_ocr = ocr_results_from_ultralytics(results[0], image_rgb)

# # Collect all extracted texts
# ocr_texts = [f"{idx+1}: {row['ocr_text']}" for idx, row in df_ocr.iterrows() if row['ocr_text']]
# summary_text = " | ".join(ocr_texts)

# # Annotate the image with bounding boxes (YOLO only)
# annotated_image = results[0].plot()

# # Add OCR text summary at the top of the image
# cv2.rectangle(annotated_image, (0,0), (annotated_image.shape[1], 50), (0,0,0), -1)  # black background
# cv2.putText(annotated_image, summary_text, (5,35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2, cv2.LINE_AA)

# # Show the result
# plt.figure(figsize=(12,12))
# plt.imshow(annotated_image)
# plt.axis("off")
# plt.title(f"Detected drugs in: {image_file}")
# plt.show()

# # Also print extracted texts
# print("Extracted OCR Texts:")
# for idx, row in df_ocr.iterrows():
#     print(f"{idx+1}: {row['ocr_text']} | Confidence = {row['ocr_confidence']:.2f} | Engine = {row['ocr_engine']}")


# **The previous code accuracy was bad**

In [ ]:
!pip install paddleocr

In [ ]:
# ============================================================
#  ocr_medicine_search.py
#  Drop this file anywhere in your Django app (e.g. ai/views_ocr.py)
#  and wire the URL below.
# ============================================================
#
#  REQUIREMENTS (add to requirements.txt):
#      pillow
#      opencv-python-headless
#      easyocr
#      pytesseract
#      imutils
#      python-Levenshtein      ← pure-Python fallback (no DB extension needed)
#      rapidfuzz               ← faster, same API, recommended
#
#  DATABASE NOTE:
#      The fuzzy search below works in TWO modes:
#
#      Mode A (recommended) — PostgreSQL pg_trgm extension
#          Run once in psql:  CREATE EXTENSION IF NOT EXISTS pg_trgm;
#          Then set:  FUZZY_BACKEND = "pg_trgm"   (see settings below)
#
#      Mode B — Python-side rapidfuzz (works with any DB: SQLite, MySQL, Postgres)
#          Set:  FUZZY_BACKEND = "rapidfuzz"
#          Slightly slower for huge catalogs (>100k rows) but zero DB setup.
#
# ============================================================

import io
import logging
import numpy as np
from PIL import Image

from django.conf import settings
from django.db import connection

from rest_framework.views import APIView
from rest_framework.parsers import MultiPartParser, FormParser
from rest_framework.permissions import IsAuthenticated
from rest_framework.response import Response
from rest_framework import status

# ── your existing Medicine model ──────────────────────────────────────────────
# Adjust the import path to match your project layout.
# From the Postman collection the catalog lives at /api/medicines/
# so the model is probably in something like:  medicines.models  or  api.models
from medicines.models import Medicine          # ← CHANGE IF NEEDED

# ── OCR pipeline (paste Cell-B code here or import from its module) ───────────
# If you saved Cell-B as  ai/ocr_pipeline.py  just do:
#   from ai.ocr_pipeline import try_rotations_and_ocr
# Otherwise paste the functions here.
from ai.ocr_pipeline import try_rotations_and_ocr   # ← CHANGE IF NEEDED

logger = logging.getLogger(__name__)

# ── tuneable constants (override in settings.py if you like) ──────────────────
FUZZY_BACKEND   = getattr(settings, "FUZZY_BACKEND",   "rapidfuzz")   # "pg_trgm" | "rapidfuzz"
MAX_CANDIDATES  = getattr(settings, "OCR_MAX_CANDIDATES", 5)           # how many matches to return
MIN_SCORE       = getattr(settings, "OCR_MIN_SCORE", 0.30)             # 0..1 — drop results below this


# ─────────────────────────────────────────────────────────────────────────────
#  HELPER: convert uploaded file → numpy RGB array
# ─────────────────────────────────────────────────────────────────────────────
def _pil_to_cv2_rgb(pil_img: Image.Image) -> np.ndarray:
    """PIL Image → uint8 numpy array in RGB order (what the OCR pipeline expects)."""
    pil_img = pil_img.convert("RGB")
    return np.array(pil_img, dtype=np.uint8)


# ─────────────────────────────────────────────────────────────────────────────
#  HELPER: run OCR on a raw numpy image, return cleaned token list
# ─────────────────────────────────────────────────────────────────────────────
def _ocr_tokens_from_image(image_rgb: np.ndarray) -> list[str]:
    """
    Runs the Cell-B OCR pipeline on the full image (no YOLO boxes needed).
    Returns a de-duplicated list of non-empty text tokens.
    """
    import cv2
    from ai.ocr_pipeline import enhance_image_for_ocr   # ← CHANGE IF NEEDED

    # We treat the whole image as one crop and run the rotation+OCR sweep
    best_text, best_conf, angle, engine = try_rotations_and_ocr(image_rgb, debug=False)

    logger.debug("OCR result | text=%r conf=%.2f angle=%s engine=%s",
                 best_text, best_conf, angle, engine)

    if not best_text:
        return []

    # Split on whitespace / newlines → individual tokens
    tokens = [t.strip() for t in best_text.split() if len(t.strip()) >= 2]
    # Also keep the full joined string as one candidate
    full = " ".join(tokens)
    candidates = list(dict.fromkeys([full] + tokens))   # dedupe, preserve order
    return candidates


# ─────────────────────────────────────────────────────────────────────────────
#  FUZZY SEARCH — pg_trgm backend
# ─────────────────────────────────────────────────────────────────────────────
def _search_pg_trgm(query: str, limit: int) -> list[dict]:
    """
    Uses PostgreSQL's trigram similarity (pg_trgm extension).
    Searches the `trade_name` column — add more columns to the similarity()
    call if your schema has e.g. `scientific_name`, `brand_name`, etc.

    Returns list of dicts: {id, name, score}
    """
    sql = """
        SELECT
            id,
            trade_name                              AS name,
            similarity(lower(trade_name), lower(%s)) AS score
        FROM medicines_medicine                     -- ← adjust table name if needed
        WHERE similarity(lower(trade_name), lower(%s)) > %s
        ORDER BY score DESC
        LIMIT %s;
    """
    with connection.cursor() as cur:
        cur.execute(sql, [query, query, MIN_SCORE, limit])
        rows = cur.fetchall()

    return [{"id": r[0], "name": r[1], "score": round(float(r[2]), 4)} for r in rows]


# ─────────────────────────────────────────────────────────────────────────────
#  FUZZY SEARCH — rapidfuzz backend (pure Python, any DB)
# ─────────────────────────────────────────────────────────────────────────────
def _search_rapidfuzz(query: str, limit: int) -> list[dict]:
    """
    Pulls medicine names from the ORM and scores them with rapidfuzz.
    Works with any database.  Fast enough for catalogs up to ~200k rows.
    """
    from rapidfuzz import fuzz, process as rfprocess

    # Fetch only id + name — minimal memory
    # Adjust field name(s) to match your actual Medicine model fields.
    # Common candidates: trade_name, name, brand_name, commercial_name
    qs = Medicine.objects.values("id", "trade_name")   # ← CHANGE FIELD NAME IF NEEDED

    choices     = {str(m["id"]): m["trade_name"] for m in qs if m["trade_name"]}
    choice_vals = list(choices.values())
    choice_keys = list(choices.keys())

    # WRatio handles partial matches, abbreviations, and case differences well
    matches = rfprocess.extract(
        query,
        choice_vals,
        scorer=fuzz.WRatio,
        limit=limit * 3,          # fetch extra, we'll filter by MIN_SCORE below
        score_cutoff=MIN_SCORE * 100,
    )
    # matches → list of (matched_string, score_0_100, index)

    results = []
    seen_ids = set()
    for matched_name, score_100, idx in matches:
        med_id  = choice_keys[idx]
        score   = score_100 / 100.0
        if score < MIN_SCORE or med_id in seen_ids:
            continue
        seen_ids.add(med_id)
        results.append({"id": int(med_id), "name": matched_name, "score": round(score, 4)})
        if len(results) >= limit:
            break

    return results


# ─────────────────────────────────────────────────────────────────────────────
#  UNIFIED SEARCH ENTRY-POINT
# ─────────────────────────────────────────────────────────────────────────────
def fuzzy_search_medicines(query: str, limit: int = MAX_CANDIDATES) -> list[dict]:
    """
    Given a text query, return up to `limit` medicines ranked by
    lexical (character-level) similarity to the query.
    """
    if not query or not query.strip():
        return []

    query = query.strip()

    if FUZZY_BACKEND == "pg_trgm":
        return _search_pg_trgm(query, limit)
    else:
        return _search_rapidfuzz(query, limit)


# ─────────────────────────────────────────────────────────────────────────────
#  THE VIEW
# ─────────────────────────────────────────────────────────────────────────────
class OCRMedicineSearchView(APIView):
    """
    POST /api/uploads/ocr-search/

    Multipart form:
        image   (file, required)   — photo of the medicine / prescription
        top_k   (int, optional)    — how many candidates to return (default 5)

    Response 200:
    {
        "ocr_raw_text": "Panadol Extra ...",
        "ocr_tokens":   ["Panadol Extra", "Panadol", "Extra"],
        "matches": [
            {"id": 42, "name": "Panadol Extra", "score": 0.94},
            {"id": 7,  "name": "Panadol",       "score": 0.81},
            ...
        ]
    }
    """
    parser_classes  = [MultiPartParser, FormParser]
    permission_classes = [IsAuthenticated]

    def post(self, request, *args, **kwargs):
        # ── 1. Validate uploaded file ─────────────────────────────────────────
        image_file = request.FILES.get("image")
        if image_file is None:
            return Response(
                {"error": "No image provided. Send a multipart field named 'image'."},
                status=status.HTTP_400_BAD_REQUEST,
            )

        top_k = int(request.data.get("top_k", MAX_CANDIDATES))
        top_k = min(max(top_k, 1), 20)   # clamp 1..20

        # ── 2. Decode image ───────────────────────────────────────────────────
        try:
            pil_img   = Image.open(io.BytesIO(image_file.read()))
            image_rgb = _pil_to_cv2_rgb(pil_img)
        except Exception as exc:
            logger.exception("Image decode failed")
            return Response(
                {"error": f"Cannot decode image: {exc}"},
                status=status.HTTP_400_BAD_REQUEST,
            )

        # ── 3. OCR ────────────────────────────────────────────────────────────
        try:
            tokens = _ocr_tokens_from_image(image_rgb)
        except Exception as exc:
            logger.exception("OCR pipeline failed")
            return Response(
                {"error": f"OCR processing failed: {exc}"},
                status=status.HTTP_500_INTERNAL_SERVER_ERROR,
            )

        if not tokens:
            return Response(
                {
                    "ocr_raw_text": "",
                    "ocr_tokens":   [],
                    "matches":      [],
                    "message":      "OCR could not extract any text from the image.",
                },
                status=status.HTTP_200_OK,
            )

        ocr_raw_text = tokens[0]   # first element is the full joined string

        # ── 4. Fuzzy DB search ────────────────────────────────────────────────
        # Strategy: try each token separately and merge results, deduplicated,
        # keeping the highest score per medicine.
        seen: dict[int, dict] = {}
        for token in tokens:
            for hit in fuzzy_search_medicines(token, limit=top_k):
                med_id = hit["id"]
                if med_id not in seen or hit["score"] > seen[med_id]["score"]:
                    seen[med_id] = hit

        # Sort by score descending, return top_k
        matches = sorted(seen.values(), key=lambda x: x["score"], reverse=True)[:top_k]

        # ── 5. Return ─────────────────────────────────────────────────────────
        return Response(
            {
                "ocr_raw_text": ocr_raw_text,
                "ocr_tokens":   tokens,
                "matches":      matches,
            },
            status=status.HTTP_200_OK,
        )


# ─────────────────────────────────────────────────────────────────────────────
#  URL WIRING  (add to your urls.py)
# ─────────────────────────────────────────────────────────────────────────────
#
#  In  api/urls.py  (or wherever /api/ routes live):
#
#      from .ocr_medicine_search import OCRMedicineSearchView
#
#      urlpatterns = [
#          ...
#          path("uploads/ocr-search/", OCRMedicineSearchView.as_view(), name="ocr-medicine-search"),
#      ]
#
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ==========================================================
# TEST OCR on a single image (random)
# ==========================================================
import matplotlib.pyplot as plt
import random
import os
import cv2

# ----------------------------------------------------------
# List of image paths (YOUR GIVEN ARRAY)
# ----------------------------------------------------------
image_paths = [
    "/content/draga-1/test/images/iphone-xs-max-164_JPG.rf.918f0b59e0d507cb77e239c2fa6ab436.jpg",
    "/content/draga-1/test/images/iphone-xs-max-196_JPG.rf.787f1fbe110018a3287f0ec4b8b5f3f6.jpg",
    "/content/draga-1/test/images/iphone-xs-max-613_JPG.rf.40092b9a8ea2a87e751b94194bf60290.jpg",
    "/content/draga-1/test/images/iphone-xs-max-668_JPG.rf.36ac06aa7e338776585a32837f7e5f86.jpg",
    "/content/draga-1/test/images/iphone-xs-max-849_JPG.rf.1e6933dc52d19e6432529b81a65612fd.jpg"
]

# Pick ONE random image from the list
image_path = random.choice(image_paths)
image_file = os.path.basename(image_path)

print(f"Testing on image: {image_file}")

# ----------------------------------------------------------
# Load image
# ----------------------------------------------------------
image = cv2.imread(image_path)
if image is None:
    raise ValueError(f"Failed to load image: {image_path}")

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# ----------------------------------------------------------
# Run YOLO detection
# ----------------------------------------------------------
results = model(image_rgb)

# ----------------------------------------------------------
# Run OCR on detected boxes
# ----------------------------------------------------------
df_ocr = ocr_results_from_ultralytics(results[0], image_rgb)

# Collect all extracted texts with rotation info
ocr_texts = []
for idx, row in df_ocr.iterrows():
    if row['ocr_text']:
        rotation_info = f" [🔄 {row['rotation_angle']}°]" if row['rotation_angle'] != 0 else ""
        ocr_texts.append(f"{idx+1}: {row['ocr_text']}{rotation_info}")

summary_text = " | ".join(ocr_texts) if ocr_texts else "No text detected"

# ----------------------------------------------------------
# Annotate image
# ----------------------------------------------------------
annotated_image = results[0].plot()

# Add OCR text summary at the top
cv2.rectangle(
    annotated_image,
    (0, 0),
    (annotated_image.shape[1], 60),
    (0, 0, 0),
    -1
)
cv2.putText(
    annotated_image,
    summary_text,
    (5, 40),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.7,
    (0, 255, 0),
    2,
    cv2.LINE_AA
)

# ----------------------------------------------------------
# Show result
# ----------------------------------------------------------
plt.figure(figsize=(14, 10))
plt.imshow(annotated_image)
plt.axis("off")
plt.title(f"Detected drugs in: {image_file}")
plt.tight_layout()
plt.show()

# ----------------------------------------------------------
# Print detailed OCR results
# ----------------------------------------------------------
print("\n" + "=" * 80)
print("EXTRACTED OCR TEXTS:")
print("=" * 80)

for idx, row in df_ocr.iterrows():
    print(f"\n{idx+1}. Text: {row['ocr_text']}")
    print(f"   Confidence: {row['ocr_confidence']:.3f} (0–1)")
    print(f"   Rotation angle: {row['rotation_angle']}°")
    print(f"   Box coords: {row['bbox']}")
    print(f"   Detection confidence: {row['box_conf']:.3f}")

print("\n" + "=" * 80)
print(f"Total detections: {len(df_ocr)}")
print(f"Successful OCR reads: {len(df_ocr[df_ocr['ocr_text'] != ''])}")
print("=" * 80)


# **search API code**



In [ ]:
pip install rapidfuzz pillow

In [ ]:
import pandas as pd
df_meds = pd.read_csv("medicines.csv")
MEDICINE_CATALOG = df_meds["trade_name"].dropna().tolist()

In [ ]:

from rapidfuzz import fuzz, process as rfprocess


MEDICINE_CATALOG = [
    "Panadol", "Panadol Extra", "Brufen", "Aspirin", "Amoxil",
    "Augmentin", "Flagyl", "Cataflam", "Voltaren", "Concor",
    "Lipitor", "Zithromax", "Cipro", "Nexium", "Omeprazole",

]

MIN_SCORE = 30

def fuzzy_search(query: str, catalog: list, top_k: int = 5) -> list:
    """ابحث عن أقرب اسم دواء معجمياً."""
    if not query or not query.strip():
        return []

    matches = rfprocess.extract(
        query,
        catalog,
        scorer=fuzz.WRatio,
        limit=top_k,
        score_cutoff=MIN_SCORE,
    )
    return [{"name": m[0], "score": round(m[1], 1)} for m in matches]



print("=" * 80)
print("FUZZY MEDICINE MATCHING RESULTS")
print("=" * 80)

all_matches = {}
for idx, row in df_ocr.iterrows():
    ocr_text = str(row['ocr_text']).strip()
    if not ocr_text:
        continue

    print(f"\n🔍 Detection #{idx+1} | OCR: '{ocr_text}' | conf: {row['ocr_confidence']:.2f}")


    tokens = list(dict.fromkeys([ocr_text] + ocr_text.split()))

    best_per_detection = {}
    for token in tokens:
        if len(token) < 2:
            continue
        hits = fuzzy_search(token, MEDICINE_CATALOG, top_k=3)
        for hit in hits:
            name = hit["name"]
            if name not in best_per_detection or hit["score"] > best_per_detection[name]:
                best_per_detection[name] = hit["score"]

    if best_per_detection:
        sorted_hits = sorted(best_per_detection.items(), key=lambda x: -x[1])
        for name, score in sorted_hits[:5]:
            print(f"   ✅ {name:30s}  score={score:.1f}/100")

            if name not in all_matches or score > all_matches[name]:
                all_matches[name] = score
    else:
        print("   ❌ No match found")


print("\n" + "=" * 80)
print("FINAL SUMMARY — Top Medicines in this image:")
print("=" * 80)
for name, score in sorted(all_matches.items(), key=lambda x: -x[1])[:10]:
    bar = "█" * int(score / 10)
    print(f"  {name:30s}  {bar:10s}  {score:.1f}/100")